# BADUML dataset pre-processing pipeline

This notebook produces audited training data to teach a vision large language model to classify UML class diagrams as GOODUML or BADUML. It extracts candidate images from student submissions, optionally audits them with an image-capable LLM via the OpenAI Batch API, consolidates audited results into an audit dataset, and formats OpenAI-style JSONL files ready for vision fine-tuning.

What it does:
- Extract images and write a raw manifest.
- Prepare and optionally submit Batch audit requests plus a UUID to source mapping.
- Process Batch output into an audit image collection and manifest.
- Split the audit set by student and write JSONL files ready for fine-tuning.

Prerequisites: 
- activated virtual environment: `source .venv/bin/activate`
- `data/ipp` submission folder
- (optional) `OPENAI_API_KEY` environment variable if planning to submit batches

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
config_path = PROJECT_ROOT / "config" / "default.json"
data_dir = PROJECT_ROOT / "data"

In [2]:
# Ensure data/clean_ipp_data.csv exists (generate if missing)

clean_csv = PROJECT_ROOT / "data" / "clean_ipp_data.csv"

if not clean_csv.is_file():
    print(f"Missing {clean_csv}. Generating via notebooks.dataset_parser.main()")
    try:
        import scripts.dataset_parser as dataset_parser
        dataset_parser.main()
        if clean_csv.is_file():
            print(f"Generated {clean_csv}")
        else:
            raise FileNotFoundError(clean_csv)
    except Exception as e:
        print(f"Failed to generate {clean_csv}: {e}")
        raise
else:
    print(f"Found {clean_csv}")

Found /home/matut/git-projects/doc-grader/data/clean_ipp_data.csv


In [3]:
import os

from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    print("OPENAI_API_KEY found in the environment.")
    OpenAI(api_key=api_key).models.list()
    print("OPENAI_API_KEY appears valid, (models.list succeeded).")
else:
    print("OPENAI_API_KEY not set, batch submission will fail if attempted.")


OPENAI_API_KEY found in the environment.
OPENAI_API_KEY appears valid, (models.list succeeded).


## Configuration

Edit these default Path variables before running if needed. `PROJECT_ROOT` is defined in the first code cell.

In [4]:
DATA_DIR      = PROJECT_ROOT / "data"  # where all data lives
TRAIN_DIR     = DATA_DIR / "vision-training" # training artefacts

IMAGES_DIR    = TRAIN_DIR / "images" # where to save BADUML/GOODUML images
MANIFEST_PATH = TRAIN_DIR / "manifest.jsonl" # metadata
MAPPING_PATH  = TRAIN_DIR / "audit_id_mapping.json"

AUDIT_DIR      = TRAIN_DIR / "audit" # where to save audited BADUML/GOODUML images
AUDIT_MANIFEST = TRAIN_DIR / "audit_manifest.jsonl"

### Extract images from student submissions

Reads `data/clean_ipp_data.csv` and locates submissions under `data/ipp`. It extracts images from PDFs and Markdown documentations via the `DocumentParser` (see `src/parsers/parser.py`) and returns `StudentImageRecord` objects saved to the manifest.

Images are resized to a maximum dimension of 2048 px, images with a short side < 128 px are ignored, and duplicates are removed by SHA1 of image bytes.

In [5]:
from scripts.baduml import extract_images
from scripts.baduml.models import save_manifest

IPP_DIR  = DATA_DIR / "ipp" # where to look for BADUML/GOODUML source files
CSV_PATH = DATA_DIR / "clean_ipp_data.csv" # cleaned dataset

records = extract_images.run(
    csv_path=CSV_PATH,
    ipp_dir=IPP_DIR,
    output_dir=IMAGES_DIR,
)
save_manifest(records, MANIFEST_PATH)
print(f"Extracted {len(records)} image records; manifest written to {MANIFEST_PATH}")

ModuleNotFoundError: No module named 'notebooks'

### Prepare (and optionally submit) an OpenAI Batch audit

Builds image batch requests and writes an `audit_id_mapping.json` that maps each UUID `custom_id` back to the source file and label. The function encodes images into base64 data URLs (the JSONL can become large) and logs a cost estimate.

Set `DRY_RUN=True` to prepare requests without submitting. If you submit (`DRY_RUN=False`), `submit_audit.submit_batch` uploads the JSONL and creates a Batch job ensure `OPENAI_API_KEY` is set. The code skips files it cannot encode and logs failures.

In [ ]:
from scripts.baduml import submit_audit

AUDIT_MODEL = "gpt-4o"
REQUESTS_PATH = PROJECT_ROOT / "notebooks" / "baduml" / "batch_audit_requests.jsonl"
DRY_RUN = True  # set to False to actually submit

n = submit_audit.write_requests_jsonl(
    images_dir=IMAGES_DIR,
    mapping_path=MAPPING_PATH,
    out_path=REQUESTS_PATH,
    model=AUDIT_MODEL,
)
print(f"Wrote {n} requests to {REQUESTS_PATH}")

if not DRY_RUN:
    result = submit_audit.submit_batch(
        requests_path=REQUESTS_PATH,
        model=AUDIT_MODEL,
        api_key_env="OPENAI_API_KEY",
    )
    if result:
        batch_id, status = result
        print(f"Submitted batch {batch_id} with status {status}")
else:
    print("Not submitting batch")

### Process Batch results into an audit dataset

Reads the Batch output JSONL and the `audit_id_mapping.json`, extracts the model `classification` and `analysis`, skipping `INVALID` results. Audited images are copied into `AUDIT_DIR/gooduml` or `AUDIT_DIR/baduml` with unique filenames to avoid collisions.

Also writes `conflicts.csv` for model/grader disagreements and `audit_manifest.jsonl` with `StudentImageRecord` entries enriched by the model analysis. Conflicting entries are still used for finetuning as the dataset size suffers greatly if not.

In [ ]:
from scripts.baduml import process_audit

# paste downloaded batch results to data/vision-training/ and rename
BATCH_RESULTS_PATH = TRAIN_DIR / "batch_output.jsonl"
CONFLICTS_CSV = TRAIN_DIR / "conflicts.csv"  # diff of audit model and human graders

process_audit.run(
    batch_results=BATCH_RESULTS_PATH,
    mapping_path=MAPPING_PATH,
    manifest_path=MANIFEST_PATH,
    audit_dir=AUDIT_DIR,
    conflicts_csv=CONFLICTS_CSV,
    audit_manifest_path=AUDIT_MANIFEST,
)
print(f"Processed batch results from {BATCH_RESULTS_PATH}")

### Format fine-tuning JSONL splits

Group records by `student_id`, then encode each image as a `data:image/...;base64,` URL and write OpenAI-style JSONL where the assistant message contains the model `analysis` and `classification` when available. Images larger than 10 MB are skipped.

Default split ratio: 0.8 train / 0.2 validation / 0.0 test

In [ ]:
from scripts.baduml import format_finetune
from scripts.baduml.models import load_manifest

FINETUNE_DIR = TRAIN_DIR / "audit_jsonl"  # directory to save audit finetuning jsonl files

audit_records = load_manifest(AUDIT_MANIFEST)
print(f"Loaded {len(audit_records)} audit records")

splits = format_finetune.split(audit_records)
format_finetune.run(splits=splits, output_dir=FINETUNE_DIR)
print(f"Wrote finetune JSONL to {FINETUNE_DIR}")

### Fine-Tuning

This notebook does not perform the Fine-Tuning itself. Within the OpenAI API dashboard, manually choose a model to fine-tune and upload the relevant `.jsonl` files.